In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")


# =========================
# ⭐ 配置区（只改这里）
# =========================
WINDOW_TITLE = "Methane Production Prediction Software 2"

# ✅ 11 个输入（删除 HAP 组 + 删除 COD/TOC）
INPUT_NAMES = [
    "C (%)", "H (%)", "O (%)", "N (%)", "S (%)",
    "Ash (%)", "Solid Content (%)",
    "HTT-T (°C)", "HTT-RT (min)",
    "AD-T (°C)", "AD-Time (day)"
]

# ✅ 2 个输出（与 Y 的两列一致）
OUTPUT_NAMES = [
    "Methane production (ml/g COD)",
    "Methane production rate (ml/(g COD·d))"
]


# =========================
# 1) 模型训练（X:0-10 共11列；Y:11-12 共2列）
# =========================
def train_model_from_excel(excel_path):
    data = pd.read_excel(excel_path)

    X = data.values[:, 0:11]
    Y = data.values[:, 11:13]

    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    # 你的 model2 最佳超参数
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=200,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    return model


# =========================
# 2) GUI
# =========================
class PredictorGUI:
    def __init__(self, root, model):
        self.root = root
        self.model = model

        self.root.title(WINDOW_TITLE)
        self.root.geometry("1050x640")

        # ===== 主容器：上(输入) + 下(输出) =====
        container = ttk.Frame(root, padding=12)
        container.pack(fill="both", expand=True)

        # 上部：输入
        input_frame = ttk.LabelFrame(container, text="Inputs", padding=12)
        input_frame.pack(fill="both", expand=True)

        # 下部：输出（放最后）
        output_frame = ttk.LabelFrame(container, text="Outputs", padding=12)
        output_frame.pack(fill="x", pady=(12, 0))

        # ===== 输入分两列容器 =====
        left_col = ttk.Frame(input_frame)
        right_col = ttk.Frame(input_frame)
        left_col.grid(row=0, column=0, sticky="nsew", padx=(0, 16))
        right_col.grid(row=0, column=1, sticky="nsew")

        input_frame.columnconfigure(0, weight=1)
        input_frame.columnconfigure(1, weight=1)

        # ===== 分组（删除 HAP 组）=====
        g1 = ttk.LabelFrame(left_col, text="Biomass characteristics", padding=10)
        g1.pack(fill="both", expand=True, pady=(0, 12))

        g2 = ttk.LabelFrame(right_col, text="HTT conditions", padding=10)
        g4 = ttk.LabelFrame(right_col, text="AD parameters", padding=10)
        g2.pack(fill="x", pady=(0, 12))
        g4.pack(fill="x")

        # ===== 创建输入框（保持顺序）=====
        self.entries = [None] * 11

        def add_row(parent, idx, label_text, r):
            ttk.Label(parent, text=label_text).grid(row=r, column=0, sticky="w", pady=6, padx=(0, 10))
            ent = ttk.Entry(parent, width=26)
            ent.grid(row=r, column=1, sticky="ew", pady=6)
            parent.columnconfigure(1, weight=1)
            self.entries[idx] = ent

        # 组1：0-6（7个）
        for r, i in enumerate(range(0, 7)):
            add_row(g1, i, INPUT_NAMES[i], r)

        # 组2：7-8（2个）
        for r, i in enumerate(range(7, 9)):
            add_row(g2, i, INPUT_NAMES[i], r)

        # 组4：9-10（2个）
        for r, i in enumerate(range(9, 11)):
            add_row(g4, i, INPUT_NAMES[i], r)

        # ===== 输出区 =====
        self.out1_var = tk.StringVar()
        self.out2_var = tk.StringVar()

        ttk.Label(output_frame, text=OUTPUT_NAMES[0]).grid(row=0, column=0, sticky="w", pady=6, padx=(0, 10))
        ttk.Entry(output_frame, textvariable=self.out1_var, state="readonly", width=45)\
            .grid(row=0, column=1, sticky="ew", pady=6)

        ttk.Label(output_frame, text=OUTPUT_NAMES[1]).grid(row=1, column=0, sticky="w", pady=6, padx=(0, 10))
        ttk.Entry(output_frame, textvariable=self.out2_var, state="readonly", width=45)\
            .grid(row=1, column=1, sticky="ew", pady=6)

        output_frame.columnconfigure(1, weight=1)

        # 按钮
        btn_frame = ttk.Frame(output_frame)
        btn_frame.grid(row=2, column=0, columnspan=2, pady=(12, 0))

        ttk.Button(btn_frame, text="Predict", command=self.predict).grid(row=0, column=0, padx=12)
        ttk.Button(btn_frame, text="Clear", command=self.clear).grid(row=0, column=1, padx=12)

        # 提示
        self.hint = tk.StringVar(value="Please input all features and click Predict")
        ttk.Label(output_frame, textvariable=self.hint, foreground="gray")\
            .grid(row=3, column=0, columnspan=2, sticky="w", pady=(10, 0))

    def _read_inputs(self):
        values = []
        for name, ent in zip(INPUT_NAMES, self.entries):
            txt = ent.get().strip()
            if txt == "":
                raise ValueError(f"{name} is empty.")
            try:
                values.append(float(txt))
            except:
                raise ValueError(f"{name} must be numeric.")
        return np.array(values, dtype=float).reshape(1, -1)

    def predict(self):
        try:
            X = self._read_inputs()
            y = self.model.predict(X)  # (1,2)
            self.out1_var.set(f"{y[0, 0]:.6f}")
            self.out2_var.set(f"{y[0, 1]:.6f}")
            self.hint.set("Prediction completed.")
        except Exception as e:
            messagebox.showerror("Prediction Error", str(e))
            self.hint.set("Prediction failed. Please check inputs.")

    def clear(self):
        for ent in self.entries:
            ent.delete(0, tk.END)
        self.out1_var.set("")
        self.out2_var.set("")
        self.hint.set("Cleared.")


# =========================
# 3) main
# =========================
def main():
    model = train_model_from_excel("model2训练集.xlsx")
    root = tk.Tk()
    PredictorGUI(root, model)
    root.mainloop()


if __name__ == "__main__":
    main()
